# Mandatory Assignment 2: Ensemble Models & Artificial Neural Networks
#### Group: 04
Group members: Brage Blomkvist, Patrick Rafael Jemieljanczyk & William Bjørling Nilsen

In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier

plt.style.use('default')
import seaborn as sns

# Save and load GridSearchCV objects:
from os import path as pth
import joblib

#Tensorflow
import tensorflow as tf

#Sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder, FunctionTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from keras import Sequential
from keras.layers import Dense, Input, Normalization
from keras.optimizers import Adam
from keras.losses import MeanSquaredError



# Models:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from keras.models import Model
from scikeras.wrappers import KerasClassifier


## Description of Dataset

The dataset used is a set of daily weather observations from Australia. The observations are taken from a span of ten years, between 2007 and 2017. 49 different locations are used to source the data. Most of the locations have a similar number of data entries, but there are some slight differences. From the Kaggle page there seems to be most difference in entries for the periods of 2007-08 and late 2012 to early 2013.

The purpose of the dataset is to see which days it is going to rain on, using the previous day's weather observations. The target value is a binary yes and no, depicting whether it rained or not on a given next day. The target value is stored in the column: "RainTomorrow" Each instance represents a day of weather observations in a time series. Each instance has a value for whether it rained on the day of the observations, and whether it rained on the next day. The threshold for rain that has been used to determine the value of both of these columns is a value of 1mm rain inside the full 24 hours of a day.

The dataset is available on Kaggle through the site publisher Joe Young (https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/data). The data was retrieved from the Australian Bureau of Meteorology and was collected daily from station observations. The dataset has missing values in multiple features and rows, and even the target value.

The dataset includes multiple guidelines that the developers utilizing the dataset should follow. Since this is a time series, chronological sampling of data should be used. Another guideline is to only fit preprocessing to training data. This will reduce the risk of leakage, since the values, such as mean, will only be derived from the training data itself. In other words, data based on test and validation partitions will not be fitted to the training data. The last guideline is to use more evaluation scores than just accuracy.

### Problem
The problem this dataset and the corresponding classification task deals with, is whether it is going to rain the following day. It is not clear if it is better for the model to falsely predict more or fewer days with rain. One way of looking at the problem is that something is supposed to happen the following day, which requires there to not be rain that day. This is the angle chosen when evaluating the models. This means that it is better to have false positives than false negatives. To limit the amount of false negatives, models which scores well in the recall metric is considered better for this problem.

In [ ]:
# Read the dataset
dataset_AUS = pd.read_csv('Dataset/weatherAUS.csv')
# Uses existing column names baked into csv file.

# Constant for target value
TARGET = "RainTomorrow"

# Show the first 5 rows to validate that the data import has been successful
dataset_AUS.head()



## Data Processing

In [ ]:
# Overview of dataset
# Look at the implementation details, data types and missing data
dataset_AUS.info()

### Missing Data

In [ ]:
# Look at the amount of missing values
dataset_AUS.isna().sum()

In [ ]:
dataset_AUS.count()

### Outlier removal

In [ ]:
#Outlier removal based on EDA
dataset_AUS = dataset_AUS.loc[(dataset_AUS['Rainfall'] < 350)
                                        & (dataset_AUS['Evaporation'] < 100)
                                        & (dataset_AUS['WindSpeed9am'] < 100)]

### Nested Cross Validation Preparation


In [ ]:
# Creating a temporary dataset
dataset_AUS_temp = dataset_AUS.copy()

# Removing rows where RainTomorrow is missing
dataset_AUS_temp = dataset_AUS_temp.dropna(subset=['RainTomorrow'])

#Converting the Date feature to datetime for sorting
dataset_AUS_temp['Date'] = pd.to_datetime(dataset_AUS_temp['Date'])

#Sorting the dataset chronologically
dataset_AUS_temp = dataset_AUS_temp.sort_values(by=['Date'])

# Temporary separating into targets and features
targets = dataset_AUS_temp['RainTomorrow'].map({'Yes' : 1, 'No' : 0})
features = dataset_AUS_temp.drop(columns=['RainTomorrow'])

# Select numerical columns
numerical_features = features.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# Select categorical columns
categorical_features = features.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

# Pipelines
to do:
   - Make the processor able to remove certain columns. Can then eventually see which models perform better

   - Can make it an option to delete the feature day_of_year

   - Make an option in the pipeline which can delete the following columns: temp3am, temp9am, year and pressure3pm, day (Need to say what Brage says bout this)


In [ ]:
# Putting the features which need to be encoded into variables as a list. To be used by pipelines to encode
location_features = ['Location']

rain_today_features = ['RainToday']

wind_features = [
    'WindGustDir',
    'WindDir9am',
    'WindDir3pm'
]

date_features = ['Date']

In [ ]:
# Cyclical encoder function
def cyclic_encoder(series, period):
    sin = np.sin(2 * np.pi * series / period)
    cos = np.cos(2 * np.pi * series / period)

    return sin, cos


In [ ]:
wind_map = {
    'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
    'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
    'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
    'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16
}

WIND_DIRECTIONS = 16
MONTH_PERIOD = 12

In [ ]:
def encode_rain_today(rain_today_dataframe):
    return rain_today_dataframe.replace({'No': 0, 'Yes': 1}).astype(float)

# encode_wind() function needs for-loop because there are three features which need to be encoded by this function.
def encode_wind(wind_data_frame):
    result = pd.DataFrame()

    for column in wind_data_frame.columns:

        wind_numbers = wind_data_frame[column].map(wind_map).astype(float)

        sin, cos = cyclic_encoder(wind_numbers, WIND_DIRECTIONS)
        result[column + '_Sine'], result[column + '_Cosine'] = sin, cos

    return result

# encode_date() function doesn't need a for loop because there is only a single feature with the dates provided.
def encode_date(dataframe):
    result = pd.DataFrame()

    date = pd.to_datetime(dataframe['Date'])
    year = date.dt.year
    month = date.dt.month
    day = date.dt.day_of_year

    is_leap_year = ((year % 4 == 0) & (year % 100 != 0)) | (year % 400 == 0)
    days_in_year = np.select([is_leap_year], [366], default = 365)

    result['Month_Sine'], result['Month_Cosine'] = cyclic_encoder(month, MONTH_PERIOD)

    result['Day_Sine'], result['Day_Cosine'] = cyclic_encoder(day, days_in_year)

    result['Year'] = year

    return result


### Encoding pipelines

In [ ]:
# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Location pipeline
location_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(target_type='binary',))
])

rain_today_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_rain_today)),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# Wind pipeline
wind_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_wind)),
    ('imputer', SimpleImputer(strategy='constant'))
])

# Date pipeline
date_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_date)),
    ('imputer', SimpleImputer(strategy='median'))
])



### Preprocessor

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('location', location_pipeline, location_features),
    ('rain_today', rain_today_pipeline, rain_today_features),
    ('wind', wind_cyclical_pipeline, wind_features),
    ('date', date_cyclical_pipeline, date_features)
])

In [ ]:
# Decision tree pipeline
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', LogisticRegression(max_iter=500))
])

# K-nearest neighbours
knn_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', KNeighborsClassifier(
        n_neighbors=5
    ))
])


# train decision tree
#dt_pipe.fit(features_train, targets_train)

## Functions for Model Training


In [ ]:
# Validation Function for Automatic Tuning
#def validation(model):
    #train_predictions = model.predict(features_train)
    #validation_predictions = model.predict(features_validation)
    #print(f'Training Accuracy: {accuracy_score(targets_train, train_predictions,):.2f} Validation: {accuracy_score(targets_validation, validation_predictions):.2f}')
    #print(f'Training Precision: {precision_score(targets_train, train_predictions):.2f} Validation: {precision_score(targets_validation, validation_predictions):.2f}')
    #print(f'Training Recall: {recall_score(targets_train, train_predictions):.2f} Validation: {recall_score(targets_validation, validation_predictions):.2f}')
    #print(f'Training F1: {f1_score(targets_train, train_predictions):.2f} Validation: {f1_score(targets_validation, validation_predictions):.2f}')

In [ ]:
# Scoring?

In [ ]:
# Function for saving and loading GridSearchCV objects
def getModelGrid(pipeline, params,features_train, targets_train, inner_splits, path="Models/default_model.pkl", overwrite_existing=0, n_jobs = 1, score = "recall"):
    # Fit model if either the flag for a new model is active or no model is found. If the flag is false and a model is found, the old model is loaded.
    if not pth.isfile(path) or overwrite_existing:
        grid = GridSearchCV(estimator=pipeline, param_grid=params, cv=inner_splits, n_jobs=n_jobs, scoring=score)
        grid.fit(features_train, targets_train)
        joblib.dump(grid, path)
        print(f"Grid saved at: {path}")
    else:
        grid = joblib.load(path)
        print(f"Grid loaded from: {path}")
    return grid, pd.DataFrame(grid.cv_results_)

## Nested Cross Validation

In [ ]:
def nested_cross_validation(
        model_pipeline,
        param_grid,
        features=features,
        targets=targets,
        outer_split=5,
        inner_split=3,
        gap = 1,
        n_jobs = 1,
        score='recall',
        save_path='Models',
        model_name = 'default',
        overwrite_existing = 0
):

    dates = pd.Series(dataset_AUS_temp['Date'])
    unique_dates = pd.Series(dataset_AUS_temp['Date'].unique())
    #Creating outer and inner CV loop
    outer_cv = TimeSeriesSplit(
        n_splits=outer_split,
        gap=gap
    )
    inner_cv = TimeSeriesSplit(
        n_splits=inner_split,
        gap=gap
        )
    assert dates.index.equals(features.index) and dates.index.equals(targets.index)
    #Storing results
    outer_results = []
    best_models = []
    best_params = []
    grid_results = []

    #Outer CV iterates on unique dates so that there won't be any mixing of testing and validation on the same day
    for fold, (trainCvIndices, testIndices) in enumerate(outer_cv.split(unique_dates), 1):

        #Saves the training+validation and test dates
        outer_train_cv_dates = unique_dates.iloc[trainCvIndices]
        outer_test_dates = unique_dates.iloc[testIndices]

        #Checking which rows belong to Training+Validation and Testing
        outer_train_cv_rows = dates.isin(outer_train_cv_dates)
        outer_test_rows = dates.isin(outer_test_dates)

        #Distributing the features and targets for CV and testing
        xTrainCv = features.loc[outer_train_cv_rows]
        yTrainCv = targets.loc[outer_train_cv_rows]

        xTest = features.loc[outer_test_rows]
        yTest = targets.loc[outer_test_rows]

        #Inner CV dates
        inner_train_cv_rows = dates.loc[outer_train_cv_rows]

        inner_splits = []
        #Inner CV splits the outer CV training dates into training and validation groups
        for inner_train_index, inner_val_index in inner_cv.split(outer_train_cv_dates):
            inner_train_dates = outer_train_cv_dates.iloc[inner_train_index]
            inner_val_dates = outer_train_cv_dates[inner_val_index]

            #Makes lists of indices that are to be included in the training and validation splits for GridSearchCV
            inner_train_rows = np.flatnonzero(
                inner_train_cv_rows.isin(
                    inner_train_dates).to_numpy()
            )
            inner_val_rows = np.flatnonzero(
                inner_train_cv_rows.isin(
                    inner_val_dates).to_numpy()
            )

            inner_splits.append((inner_train_rows, inner_val_rows))

        grid, grid_result = getModelGrid(
            pipeline=model_pipeline,
            params=param_grid,
            features_train=xTrainCv,
            targets_train=yTrainCv,
            inner_splits=inner_splits,
            path=f"{save_path}/{model_name}_fold_{fold}.pkl",
            overwrite_existing=overwrite_existing,
            n_jobs=n_jobs,
            score=score,
        )

        grid_results.append(grid_result)

        print(
                f"Fold {fold}: "
                f"Training: {outer_train_cv_dates.min()} to {outer_train_cv_dates.max()}, "
                f"Testing: {outer_test_dates.min()} to {outer_test_dates.max()}"
        )

        #Finding the best performing models and their parameters
        predictions = grid.best_estimator_.predict(xTest)

        #Evaluation
        results = {
            "Fold": fold,
            "Accuracy": accuracy_score(yTest, predictions),
            "Precision": precision_score(yTest, predictions, zero_division=0),
            "Recall": recall_score(yTest, predictions, zero_division=0),
            "f1": f1_score(yTest, predictions, zero_division=0),
        }

        outer_results.append(results)
        best_models.append(grid.best_estimator_)
        best_params.append(grid.best_params_)

    return outer_results, best_models, best_params, grid_results

In [ ]:
# Demo of nested_cross_validation()
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])
dt_params = {
    "estimator__max_depth": [12, 15, 20, 25],
    "estimator__max_features": [None, 12, 20, 25],
    "estimator__min_samples_split": [0.005, 0.015, 0.02, 20, 25],
    "estimator__min_samples_leaf": [0.015, 0.025, 0.035, 15, 20]
}
outer_results, best_models, best_params, grid_results = nested_cross_validation(dt_pipe, dt_params, features, targets, n_jobs=-1, save_path="Models/Baseline", model_name="dt_pipe")

In [ ]:
def printResults(results_list):
    for results in results_list:
        for key, value in results.items():
            print(f"{key}: {value:.3f}")

In [ ]:
printResults(outer_results)

## Baseline Models

### Decision Trees
The decision tree baseline model is used with automated tuning. This model is made to showcase some of the steps taken during development. It was developed before nested cross validation was implemented. As such, it is deprecated and will be updated before the exam folder assignment. Multiple parts of the process has been commented out since the temporary splitting technique was removed.

In [ ]:
# Set a flag for whether a new model is trained or an old is loaded.
dt_new_flag = 0

In [ ]:
# Pipeline for decision tree
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])
# Shows parameters within the pipeline which can be tuned.
dt_pipe.get_params()

For all the automated tuning, plots are used to see whether the range of parameters chosen is enough to cover the parameters which yields the best result. This process is started by using an initial broad range of parameters, as shown below. The process is only shown for this model type. For other models, only the last parameters and check is provided. Note, more than two runs have been made, but only two are shown as saving more would clutter the notebook.

In [ ]:
# Initial parameters for decision tree to check which are tuned for decision trees.
dt_params_init = {
    "estimator__max_depth": [2, 5, 15, 20, 25],
    "estimator__max_features": [None,"sqrt", 12, 20, 25],
    "estimator__criterion": ["gini", "entropy"],
    "estimator__min_samples_split": [0.05, 0.3, 3, 8, 12],
    "estimator__min_samples_leaf": [0.05, 0.3, 3, 8, 12]
}


In [ ]:
# Old way of using getModelGrid before nestedCV was added.
# fit initial model to investigate parameters.
# dt_grid_init, dt_cv_results_init = getModelGrid(dt_pipe, dt_params_init, path="Models/Baseline/dt_pipe_init.pkl", new_flag=dt_new_flag, n_jobs=8)

# New temporary way which simply fetches the deprecated model.
path = "Models/Baseline/dt_pipe_init.pkl"
dt_grid_init = joblib.load(path)
print(f"Grid loaded from: {path}")
dt_cv_results_init = pd.DataFrame(grid.cv_results_)

In [ ]:
# Plots for checking if appropriate parameter values have been covered in the tuning.
# Each parameter has its own plot.
fig, ax = plt.subplots(2, 3, figsize=(20, 10))
# Values of None are passed in as a string for the violin plot to work.
dt_cv_results_init['param_estimator__max_features'] = dt_cv_results_init['param_estimator__max_features'].fillna('None')
sns.violinplot(data=dt_cv_results_init, x='param_estimator__max_features', y='mean_test_score', ax=ax[0, 0])
sns.lineplot(data=dt_cv_results_init, x='param_estimator__max_depth', y='mean_test_score', ax=ax[0, 1])
sns.barplot(data=dt_cv_results_init, x='param_estimator__criterion', y='mean_test_score', ax=ax[0, 2])
sns.lineplot(data=dt_cv_results_init, x='param_estimator__min_samples_split', y='mean_test_score', ax=ax[1, 0])
sns.lineplot(data=dt_cv_results_init, x='param_estimator__min_samples_leaf', y='mean_test_score', ax=ax[1, 1])

In [ ]:
# Get scorer used by GridSearchCV
dt_grid_init.scorer_

In [ ]:
# Validate tuning
# validation(dt_grid_init)

In [ ]:
# Final set of parameters to check with GridSearchCV function.
dt_params = {
    "estimator__max_depth": [12, 15, 20, 25],
    "estimator__max_features": [None, 25, 26, 28],
    "estimator__min_samples_split": [0.005, 0.015, 18, 19, 20, 21, 22],
    "estimator__min_samples_leaf": [0.005, 0.015, 13, 14, 15, 16, 17],
    "estimator__criterion": ["entropy"]
}

In [ ]:

# Old way of using getModelGrid before nestedCV was added.
# Fit or load final dt model.
# dt_grid, dt_cv_results  = getModelGrid(dt_pipe, dt_params, path="Models/Baseline/dt_pipe.pkl", new_flag=dt_new_flag, n_jobs=8)

# New temporary way which simply fetches the deprecated model.
path = "Models/Baseline/dt_pipe.pkl"
dt_grid = joblib.load(path)
print(f"Grid loaded from: {path}")
dt_cv_results = pd.DataFrame(grid.cv_results_)


In [ ]:
# Plots for checking if appropriate parameter values have been covered in the tuning.
# Each parameter has its own plot.
fig, ax = plt.subplots(2, 2, figsize=(10, 10))
# Values of None are passed in as a string for the violin plot to work.
dt_cv_results['param_estimator__max_features'] = dt_cv_results['param_estimator__max_features'].fillna('None')
sns.violinplot(data=dt_cv_results, x='param_estimator__max_features', y='mean_test_score', ax=ax[0, 0])
sns.lineplot(data=dt_cv_results, x='param_estimator__max_depth', y='mean_test_score', ax=ax[0, 1])
sns.lineplot(data=dt_cv_results, x='param_estimator__min_samples_split', y='mean_test_score', ax=ax[1, 0])
sns.lineplot(data=dt_cv_results, x='param_estimator__min_samples_leaf', y='mean_test_score', ax=ax[1, 1])

While the model performance may be better for higher values of max depth, it will cause overfitting. Therefore, a model needs to be tuned with a balanced max depth. This is not done for this assignment, but will hopefully be improved by using nested cross validation before the final folder submission.

In [ ]:
# validation(dt_grid)
# Since nested cross validation isn't implemented, the model will only tune to training and get overfitted with high max depth.

In [ ]:
# (Automated tuning)

## Advanced Models

### Random Forest

In [ ]:
# Flag for creating new model
rf_new_flag = 0

In [ ]:
# Pipeline for random forest
rf_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', RandomForestClassifier())
])
# Shows parameters within the pipeline which can be tuned.
rf_pipe.get_params()

In [ ]:
# Random forrest estimator parameters
# To limit runtime when creating the model, only parameters which had the most impact on the decision tree test score are tuned.
# Additionally, the decision tree tuning is used as a basis for which values are used in the tuning.

# Only the last tuned parameters are shown.
# A full list of the tested parameters which are not in the final tuning is found below:
#   "estimator__n_estimators": [50, 100, 150, 200], Note: As 100 is the default value, it is removed from the last test
#   "estimator__max_depth": [10, 20],
#   "estimator__min_samples_split": [18, 20],
#   "estimator__min_samples_leaf": [8, 10, 11, 12, 13]
rf_params = {
    "estimator__max_depth": [15, 25, None],
    "estimator__max_features": [None],
    "estimator__criterion": ["entropy"],
    "estimator__min_samples_split": [14, 15, 16],
    "estimator__min_samples_leaf": [5, 6, 7]
}


In [ ]:
rf_scores, rf_best_models, rf_best_params, rf_cv_results  = nested_cross_validation(rf_pipe,
                                                  rf_params,
                                                  model_name="rf_pipe2",
                                                  save_path="Models/Advanced",
                                                  overwrite_existing=rf_new_flag,
                                                  n_jobs=8)
rf_cv_results

In [ ]:
# Check if enough parameters are covered in the tuning.
# This only shows the last tuning done.
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
# Handles None as parameter for max depth by using violin plot with None as a string.
rf_cv_results['param_estimator__max_depth'] = rf_cv_results['param_estimator__max_depth'].fillna('None')
sns.violinplot(data=rf_cv_results, x='param_estimator__max_depth', y='mean_test_score', ax=ax[0])
sns.lineplot(data=rf_cv_results, x='param_estimator__min_samples_split', y='mean_test_score', ax=ax[1])
sns.lineplot(data=rf_cv_results, x='param_estimator__min_samples_leaf', y='mean_test_score', ax=ax[2])

In [ ]:
printResults(rf_scores)

## Advanced Neural Networks

### KerasClassifier model

In [ ]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

def create_nn(meta, num_hidden1=20, num_hidden2=10):

    #Gets number of featuress for input shape
    n_features = meta["n_features_in_"]
    input = Input(shape=(n_features,))
    hidden1 = Dense(num_hidden1, activation='relu')(input)
    hidden2 = Dense(num_hidden2, activation='relu')(input) + Dense(num_hidden2, activation='relu')(hidden1)
    output = Dense(1, activation='sigmoid')(hidden2)

    model = Model(inputs=input, outputs=output)

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

nn_classifier = KerasClassifier(
    model=create_nn,
    model__num_hidden1=20,
    model__num_hidden2=10,
    epochs=50,
)

nn2_pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("estimator", nn_classifier)
])

test_params = {
    "estimator__model__num_hidden1": [20],
    "estimator__model__num_hidden2": [10]
}

kc_test_scores, kc_test_best_models, kc_test_best_params, kc_test_grid_results = nested_cross_validation(
    model_pipeline=nn2_pipe,
    model_name="nn2_pipe_test",
    param_grid=test_params,
    features=features,
    targets=targets,
    outer_split=2,
    inner_split=2,
    gap=1,
    score="recall",
    save_path="Models/Neural_Networks",
    overwrite_existing=0
)

In [ ]:
printResults(kc_test_scores)

In [ ]:
nn2_params = {
    "estimator__model__num_hidden1": [20, 30, 40],
    "estimator__model__num_hidden2": [5, 10, 20]
}
nn2_scores, best_models, best_params, grid_results = nested_cross_validation(
    model_pipeline=nn2_pipe,
    model_name="nn2_pipe_",
    param_grid=nn2_params,
    features=features,
    targets=targets,
    outer_split=5,
    inner_split=3,
    gap=1,
    score="recall",
    save_path="Models/Neural_Networks",
    overwrite_existing=0
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
kc_cv_results = pd.concat(grid_results, ignore_index=True)
sns.lineplot(data=kc_cv_results, x='param_estimator__model__num_hidden1', y='mean_test_score', ax=ax[0])
sns.lineplot(data=kc_cv_results, x='param_estimator__model__num_hidden2', y='mean_test_score', ax=ax[1])



In [ ]:
printResults(nn2_scores)

### MLPClassifier model

In [ ]:
nnMLP_pipe = Pipeline([
    ("preprocessing", preprocessor),
    ("estimator", MLPClassifier(max_iter=700, early_stopping=False, tol=1e-5))])

#Initial parameters for tuning
#mlp_params = {
#    'estimator__hidden_layer_sizes': [(20,), (20, 10), (20, 15, 10, 5)],
#    'estimator__alpha': [0.001, 0.01, 0.05, 0.1],
#    'estimator__learning_rate_init': [0.001, 0.01, 0.05, 0.1]
#}
mlp_params = {
    'estimator__hidden_layer_sizes': [(40,), (40,20), (30,), (20,), (20, 10)],
    'estimator__alpha': [1e-8,1e-7,1e-4, 0.01, 0.1],
    'estimator__learning_rate_init': [1e-4,0.001, 0.01, 0.1]
}

mlp_scores, best_mlp_models, best_mlp_params, mlp_grid_results = nested_cross_validation(
    model_pipeline=nnMLP_pipe,
    model_name="nnMLP_pipe",
    param_grid=mlp_params,
    features=features,
    targets=targets,
    outer_split=5,
    inner_split=3,
    n_jobs=-1,
    gap=1,
    score="recall",
    save_path="Models/Neural_Networks",
    overwrite_existing=0
)
mlp_grid_results


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
mlp_cv_results = pd.concat(mlp_grid_results, ignore_index=True)

layer_order = ["(40,)","(40, 20)","(30,)", "(20,)", "(20, 10)"]
mlp_cv_results["layers"] = pd.Categorical(
    mlp_cv_results["param_estimator__hidden_layer_sizes"].astype(str),
    categories=layer_order)

sns.violinplot(data=mlp_cv_results, x='param_estimator__alpha', y='mean_test_score', ax=ax[0])
sns.violinplot(data=mlp_cv_results, x='param_estimator__learning_rate_init', y='mean_test_score', ax=ax[1])
sns.lineplot(data=mlp_cv_results, x='layers', y='mean_test_score', ax=ax[2])



## Evaluation & Comparison

### Testing

#### Combined Results Table

### Confusion Matrices & Additional Plots

### Conclusion